# 综合案例：电影推荐系统实战

本 Notebook 旨在构建一个完整的电影推荐系统。

我们将按照以下流程进行：
1. **数据加载与清洗**
2. **探索性数据分析 (EDA)**：理解长尾分布、稀疏性等。
3. **模型构建与对比**：
    - **Baseline**: 统计学基线 (Global/User/Item Mean)。
    - **Collaborative Filtering**: UserCF 和 ItemCF。
    - **Matrix Factorization**: SVD。
4. **评估**: 使用 RMSE 评估各模型性能。

In [ ]:
# 安装依赖库 (如果尚未安装)
!pip install numpy pandas matplotlib seaborn scikit-surprise scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from surprise import Dataset, Reader, SVD, KNNBasic, Accuracy
from surprise.model_selection import PredefinedKFold

# 设置绘图风格
sns.set_style("whitegrid")
%matplotlib inline

## 1. 数据加载

我们要加载训练集和测试集。数据集格式为 `userId,movieId,rating,timestamp`。

In [ ]:
# 加载数据
# 注意：请确保当前目录下存在 recommendation-ratings-train.txt 和 recommendation-ratings-test.txt
train_file = 'recommendation-ratings-train.txt'
test_file = 'recommendation-ratings-test.txt'

names = ['userId', 'movieId', 'rating', 'timestamp']
train_df = pd.read_csv(train_file, names=names, header=0)
test_df = pd.read_csv(test_file, names=names, header=0)

print(f"训练集大小: {train_df.shape}")
print(f"测试集大小: {test_df.shape}")
train_df.head()

## 2. 探索性数据分析 (EDA)

在建模前，我们需要了解数据的基本特征：稀疏度、长尾效应和评分分布。

In [ ]:
# 2.1 计算稀疏度 (Sparsity)
n_users = train_df['userId'].nunique()
n_items = train_df['movieId'].nunique()
n_ratings = len(train_df)

total_possible = n_users * n_items
sparsity = 1 - (n_ratings / total_possible)

print(f"用户数: {n_users}, 电影数: {n_items}, 评分数: {n_ratings}")
print(f"矩阵稀疏度: {sparsity:.4%}")

In [ ]:
# 2.2 长尾效应 (Long Tail)
# 统计每部电影的评分次数
item_counts = train_df['movieId'].value_counts().values

plt.figure(figsize=(10, 5))
plt.plot(item_counts)
plt.title('Long Tail Distribution (Item Popularity)')
plt.xlabel('Items (Sorted by Popularity)')
plt.ylabel('Number of Ratings')
plt.show()

In [ ]:
# 2.3 评分分布
plt.figure(figsize=(8, 5))
sns.countplot(x='rating', data=train_df)
plt.title('Rating Distribution')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.show()

## 3. 模型构建与对比

我们将对比三种类型的模型：
1. **统计学基线 (Statistical Baselines)**: Pandas 实现
2. **协同过滤 (Collaborative Filtering)**: Surprise 实现
3. **矩阵分解 (Matrix Factorization)**: Surprise 实现

为了统一评估标准，我们将计算 **RMSE**。

In [ ]:
# 准备 Surprise 数据格式
# Surprise 支持从文件加载训练集和测试集 (PredefinedKFold)
# 但为了灵活，我们这里使用 Dataset.load_from_df 加载训练集，手动构建测试集列表

reader = Reader(rating_scale=(0.5, 5.0))

# 1. 构建训练集 (Trainset Object)
data_train = Dataset.load_from_df(train_df[['userId', 'movieId', 'rating']], reader)
trainset = data_train.build_full_trainset()

# 2. 构建测试集 (List of Tuples)
testset = list(test_df[['userId', 'movieId', 'rating']].itertuples(index=False, name=None))

print("Surprise 数据集构建完成。")

### 3.1 统计学基线 (Baselines)

我们先用简单的统计量作为基准。

In [ ]:
from sklearn.metrics import mean_squared_error

def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# 1. Global Mean
global_mean = train_df['rating'].mean()
test_df['pred_global'] = global_mean
rmse_global = calculate_rmse(test_df['rating'], test_df['pred_global'])
print(f"Global Mean RMSE: {rmse_global:.4f}")

# 2. User Mean
user_means = train_df.groupby('userId')['rating'].mean()
# 映射到测试集，如果用户在训练集中不存在(冷启动)，则使用全局平均
test_df['pred_user'] = test_df['userId'].map(user_means).fillna(global_mean)
rmse_user = calculate_rmse(test_df['rating'], test_df['pred_user'])
print(f"User Mean RMSE:   {rmse_user:.4f}")

# 3. Item Mean
item_means = train_df.groupby('movieId')['rating'].mean()
test_df['pred_item'] = test_df['movieId'].map(item_means).fillna(global_mean)
rmse_item = calculate_rmse(test_df['rating'], test_df['pred_item'])
print(f"Item Mean RMSE:   {rmse_item:.4f}")

### 3.2 & 3.3 协同过滤与矩阵分解 (Surprise)

使用 Surprise 库实现 UserCF, ItemCF 和 SVD。

In [ ]:
# 定义一个通用的评估函数
def evaluate_algo(algo, name):
    print(f"\n正在训练 {name} ...")
    algo.fit(trainset)
    predictions = algo.test(testset)
    rmse = Accuracy.rmse(predictions, verbose=False)
    print(f"{name} RMSE: {rmse:.4f}")
    return rmse

results = {}

# 1. UserCF (基于用户的协同过滤)
sim_options = {'name': 'cosine', 'user_based': True}
algo_usercf = KNNBasic(sim_options=sim_options, verbose=False)
results['UserCF'] = evaluate_algo(algo_usercf, 'UserCF')

# 2. ItemCF (基于物品的协同过滤)
sim_options = {'name': 'cosine', 'user_based': False}
algo_itemcf = KNNBasic(sim_options=sim_options, verbose=False)
results['ItemCF'] = evaluate_algo(algo_itemcf, 'ItemCF')

# 3. SVD (矩阵分解)
algo_svd = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02)
results['SVD'] = evaluate_algo(algo_svd, 'SVD')

## 4. 结果汇总

我们将所有模型的 RMSE 汇总对比。

In [ ]:
# 添加基线结果
results['Global Mean'] = rmse_global
results['User Mean'] = rmse_user
results['Item Mean'] = rmse_item

# 转换为 DataFrame 展示
df_results = pd.DataFrame(list(results.items()), columns=['Model', 'RMSE'])
df_results = df_results.sort_values('RMSE')

print("最终评估结果 (RMSE 越低越好):")
display(df_results)

# 可视化
plt.figure(figsize=(10, 6))
sns.barplot(x='RMSE', y='Model', data=df_results, palette='viridis')
plt.title('Model Comparison (RMSE)')
plt.xlabel('RMSE (Lower is Better)')
plt.show()